# Práctica 2 · Similitud de documentos con Bag-of-Words y similitud del coseno

In [1]:
from pathlib import Path
from docx import Document
from nltk.corpus import stopwords
import re
import nltk
import torch
import torch.nn.functional as F


In [2]:
carpeta = Path.cwd()
archivos = sorted(carpeta.glob("doc*.docx"))

esperados = [f"doc{i:02d}.docx" for i in range(1, 13)]
nombres_encontrados = [ruta.name for ruta in archivos]
if nombres_encontrados != esperados:
    raise FileNotFoundError(
        f"Se esperaban los archivos {esperados} en {carpeta}. "
        f"Se encontraron: {nombres_encontrados}."
    )

documentos = []
nombres = []
for ruta in archivos:
    documento_word = Document(ruta)
    texto = " ".join(
        parrafo.text for parrafo in documento_word.paragraphs
        if parrafo.text.strip()
    )
    if not texto.strip():
        raise ValueError(f"El documento {ruta.name} no contiene texto en párrafos.")
    documentos.append(texto)
    nombres.append(ruta.name)

print(f"Documentos cargados ({len(nombres)}): {nombres}")

Documentos cargados (12): ['doc01.docx', 'doc02.docx', 'doc03.docx', 'doc04.docx', 'doc05.docx', 'doc06.docx', 'doc07.docx', 'doc08.docx', 'doc09.docx', 'doc10.docx', 'doc11.docx', 'doc12.docx']


In [3]:
nltk.download("stopwords", quiet=True)

STOPWORDS = set(stopwords.words("spanish"))

def preprocesar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-záéíóúñü\s]", " ", texto)
    palabras = texto.split()
    palabras = [p for p in palabras if p not in STOPWORDS]
    return palabras

docs_tokenizados = [preprocesar(doc) for doc in documentos]

for nombre, tokens in zip(nombres, docs_tokenizados):
    print(f"{nombre}: {len(tokens)} palabras tras preprocesado")

doc01.docx: 58 palabras tras preprocesado
doc02.docx: 51 palabras tras preprocesado
doc03.docx: 47 palabras tras preprocesado
doc04.docx: 44 palabras tras preprocesado
doc05.docx: 42 palabras tras preprocesado
doc06.docx: 46 palabras tras preprocesado
doc07.docx: 44 palabras tras preprocesado
doc08.docx: 43 palabras tras preprocesado
doc09.docx: 45 palabras tras preprocesado
doc10.docx: 42 palabras tras preprocesado
doc11.docx: 74 palabras tras preprocesado
doc12.docx: 83 palabras tras preprocesado


In [4]:
vocabulario = sorted(set(palabra for tokens in docs_tokenizados for palabra in tokens))
palabra_a_indice = {palabra: i for i, palabra in enumerate(vocabulario)}

print(f"Tamaño del vocabulario: {len(vocabulario)} palabras")

def vectorizar_bow(tokens, vocab_index):
    vector = torch.zeros(len(vocab_index))
    for palabra in tokens:
        if palabra in vocab_index:
            vector[vocab_index[palabra]] += 1
    return vector

matriz_bow = torch.stack([vectorizar_bow(tokens, palabra_a_indice) for tokens in docs_tokenizados])
print(f"Forma de la matriz BoW: {matriz_bow.shape}")

Tamaño del vocabulario: 411 palabras
Forma de la matriz BoW: torch.Size([12, 411])


In [5]:
n_docs = matriz_bow.shape[0]
similitud_bow = torch.zeros((n_docs, n_docs))

for i in range(n_docs):
    for j in range(n_docs):
        similitud_bow[i, j] = F.cosine_similarity(
            matriz_bow[i], matriz_bow[j], dim=0
        )

print("Matriz de similitud del coseno (BoW):")
for i in range(n_docs):
    fila = " ".join(f"{similitud_bow[i, j]:.2f}" for j in range(n_docs))
    print(f"{nombres[i]}: {fila}")

Matriz de similitud del coseno (BoW):
doc01.docx: 1.00 0.04 0.00 0.08 0.10 0.14 0.06 0.02 0.00 0.05 0.15 0.09
doc02.docx: 0.04 1.00 0.10 0.00 0.00 0.10 0.12 0.03 0.00 0.11 0.19 0.11
doc03.docx: 0.00 0.10 1.00 0.00 0.00 0.03 0.10 0.02 0.00 0.09 0.00 0.05
doc04.docx: 0.08 0.00 0.00 1.00 0.10 0.07 0.06 0.13 0.22 0.00 0.14 0.25
doc05.docx: 0.10 0.00 0.00 0.10 1.00 0.06 0.00 0.19 0.21 0.04 0.11 0.06
doc06.docx: 0.14 0.10 0.03 0.07 0.06 1.00 0.13 0.04 0.02 0.08 0.13 0.10
doc07.docx: 0.06 0.12 0.10 0.06 0.00 0.13 1.00 0.02 0.00 0.06 0.19 0.15
doc08.docx: 0.02 0.03 0.02 0.13 0.19 0.04 0.02 1.00 0.09 0.02 0.09 0.03
doc09.docx: 0.00 0.00 0.00 0.22 0.21 0.02 0.00 0.09 1.00 0.04 0.17 0.32
doc10.docx: 0.05 0.11 0.09 0.00 0.04 0.08 0.06 0.02 0.04 1.00 0.07 0.09
doc11.docx: 0.15 0.19 0.00 0.14 0.11 0.13 0.19 0.09 0.17 0.07 1.00 0.30
doc12.docx: 0.09 0.11 0.05 0.25 0.06 0.10 0.15 0.03 0.32 0.09 0.30 1.00


In [6]:
pares = []
for i in range(n_docs):
    for j in range(i + 1, n_docs):
        pares.append((similitud_bow[i, j].item(), nombres[i], nombres[j]))

pares_ordenados = sorted(pares, key=lambda x: x[0], reverse=True)

print("Pares más similares (BoW):")
for sim, a, b in pares_ordenados[:5]:
    print(f"  {a} - {b}: {sim:.3f}")

print("Pares menos similares (BoW):")
for sim, a, b in pares_ordenados[-5:]:
    print(f"  {a} - {b}: {sim:.3f}")

Pares más similares (BoW):
  doc09.docx - doc12.docx: 0.323
  doc11.docx - doc12.docx: 0.302
  doc04.docx - doc12.docx: 0.249
  doc04.docx - doc09.docx: 0.217
  doc05.docx - doc09.docx: 0.207
Pares menos similares (BoW):
  doc03.docx - doc09.docx: 0.000
  doc03.docx - doc11.docx: 0.000
  doc04.docx - doc10.docx: 0.000
  doc05.docx - doc07.docx: 0.000
  doc07.docx - doc09.docx: 0.000


In [7]:
for nombre, texto in zip(nombres, documentos):
    extracto = " ".join(texto.split())[:300]
    print(f"{nombre}: {extracto}{'…' if len(texto) > 300 else ''}\n")

print("Términos compartidos en los cinco pares más similares:")
for sim, a, b in pares_ordenados[:5]:
    ia, ib = nombres.index(a), nombres.index(b)
    comunes = set(docs_tokenizados[ia]) & set(docs_tokenizados[ib])
    ordenados = sorted(
        comunes,
        key=lambda p: (-(docs_tokenizados[ia].count(p) * docs_tokenizados[ib].count(p)), p)
    )
    print(f"{a} – {b} ({sim:.3f}): {', '.join(ordenados[:15])}")

doc01.docx: Nuevo smartphone de alta gama promete revolucionar el mercado Hoy se anunció el lanzamiento del último smartphone de la reconocida marca Tecnophone. Este nuevo dispositivo, denominado Tecnophone XZ, promete características sobresalientes que seguramente causarán revuelo en la industria móvil. Incorp…

doc02.docx: La duración de la batería en smartphones: un desafío constante La vida útil de la batería se ha convertido en uno de los temas más discutidos en el mundo de los smartphones. Con la constante demanda de aplicaciones y servicios que consumen grandes cantidades de energía, los fabricantes están bajo pr…

doc03.docx: Las aplicaciones móviles más descargadas del mes El auge de las aplicaciones móviles continúa creciendo a pasos agigantados. Este mes, la aplicación de edición de video ClipMaster ha liderado las listas de descargas, seguida de cerca por la app de fitness FitJourney. Ambas aplicaciones demuestran el…

doc04.docx: El futuro de los coches eléctricos brilla m

In [8]:
N = matriz_bow.shape[0]

total_palabras_doc = matriz_bow.sum(dim=1, keepdim=True)
tf = matriz_bow / total_palabras_doc

df = (matriz_bow > 0).sum(dim=0)

idf = torch.log(torch.tensor(N, dtype=torch.float32) / df.float())

matriz_tfidf = tf * idf

print(f"Forma de la matriz TF-IDF: {matriz_tfidf.shape}")

Forma de la matriz TF-IDF: torch.Size([12, 411])


In [9]:
similitud_tfidf = torch.zeros((N, N))

for i in range(N):
    for j in range(N):
        similitud_tfidf[i, j] = F.cosine_similarity(
            matriz_tfidf[i],
            matriz_tfidf[j], dim=0
        )

print("Matriz de similitud del coseno (TF-IDF):")

for i in range(N):
    fila = " ".join(f"{similitud_tfidf[i, j]:.2f}" for j in range(N))
    print(f"{nombres[i]}: {fila}")

Matriz de similitud del coseno (TF-IDF):
doc01.docx: 1.00 0.01 0.00 0.04 0.05 0.06 0.00 0.01 0.00 0.01 0.04 0.02
doc02.docx: 0.01 1.00 0.06 0.00 0.00 0.03 0.03 0.02 0.00 0.04 0.07 0.04
doc03.docx: 0.00 0.06 1.00 0.00 0.00 0.01 0.06 0.01 0.00 0.04 0.00 0.02
doc04.docx: 0.04 0.00 0.00 1.00 0.03 0.03 0.03 0.06 0.07 0.00 0.05 0.09
doc05.docx: 0.05 0.00 0.00 0.03 1.00 0.02 0.00 0.07 0.07 0.02 0.05 0.02
doc06.docx: 0.06 0.03 0.01 0.03 0.02 1.00 0.03 0.01 0.00 0.01 0.03 0.03
doc07.docx: 0.00 0.03 0.06 0.03 0.00 0.03 1.00 0.00 0.00 0.01 0.04 0.04
doc08.docx: 0.01 0.02 0.01 0.06 0.07 0.01 0.00 1.00 0.03 0.01 0.03 0.02
doc09.docx: 0.00 0.00 0.00 0.07 0.07 0.00 0.00 0.03 1.00 0.02 0.07 0.13
doc10.docx: 0.01 0.04 0.04 0.00 0.02 0.01 0.01 0.01 0.02 1.00 0.01 0.02
doc11.docx: 0.04 0.07 0.00 0.05 0.05 0.03 0.04 0.03 0.07 0.01 1.00 0.12
doc12.docx: 0.02 0.04 0.02 0.09 0.02 0.03 0.04 0.02 0.13 0.02 0.12 1.00


In [10]:
pares_tfidf = []

for i in range(N):
    for j in range(i + 1, N):
        pares_tfidf.append(
            (
                similitud_tfidf[i, j].item(),
                nombres[i],
                nombres[j]
            )
        )

pares_tfidf_ordenados = sorted(
    pares_tfidf,
    key=lambda x: x[0],
    reverse=True
)

print("Pares más similares (TF-IDF):")
for sim, a, b in pares_tfidf_ordenados[:5]:
    print(f"  {a} - {b}: {sim:.3f}")

print("\nPares menos similares (TF-IDF):")
for sim, a, b in pares_tfidf_ordenados[-5:]:
    print(f"  {a} - {b}: {sim:.3f}")

Pares más similares (TF-IDF):
  doc09.docx - doc12.docx: 0.132
  doc11.docx - doc12.docx: 0.117
  doc04.docx - doc12.docx: 0.091
  doc05.docx - doc08.docx: 0.072
  doc09.docx - doc11.docx: 0.071

Pares menos similares (TF-IDF):
  doc03.docx - doc09.docx: 0.000
  doc03.docx - doc11.docx: 0.000
  doc04.docx - doc10.docx: 0.000
  doc05.docx - doc07.docx: 0.000
  doc07.docx - doc09.docx: 0.000


In [11]:
print("Comparación de las 5 parejas más similares")
print("\nBoW:")
for sim, a, b in pares_ordenados[:5]:
    print(f"  {a} - {b}: {sim:.3f}")

print("\nTF-IDF:")
for sim, a, b in pares_tfidf_ordenados[:5]:
    print(f"  {a} - {b}: {sim:.3f}")

Comparación de las 5 parejas más similares

BoW:
  doc09.docx - doc12.docx: 0.323
  doc11.docx - doc12.docx: 0.302
  doc04.docx - doc12.docx: 0.249
  doc04.docx - doc09.docx: 0.217
  doc05.docx - doc09.docx: 0.207

TF-IDF:
  doc09.docx - doc12.docx: 0.132
  doc11.docx - doc12.docx: 0.117
  doc04.docx - doc12.docx: 0.091
  doc05.docx - doc08.docx: 0.072
  doc09.docx - doc11.docx: 0.071
